In [ ]:
# Revised Universal Soil Loss Equation (RUSLE)

### Final Script using USSURGO Database Kffact attribute to calculate K-factor

In [1]:
# K (US units) from gSSURGO using kffact with fallback:
#  - First: kffact (0–25 cm depth-weighted → component-weighted → MUKEY)
#  - Then: fill NaNs with kwfact (same method)
#  - Finally: any remaining NaNs that are WATER map units → set to 0
#  - Others remain NaN
#
# Deps: pandas, geopandas, numpy, rasterio, rioxarray, shapely

import os
import math
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.transform import from_origin
from rasterio.features import rasterize
from shapely.geometry import mapping



muaggatt_csv  = "muaggatt.csv"    # OPTIONAL fallback for missing K (mapunit-level)


# ========= USER INPUTS =========
component_csv = "compct.csv"
chorizon_csv  = "chorizon.csv"
mapunit_csv   = "mapunit.csv"   # used only to identify WATER map units

aoi_shp       = "output/ssurgo_soil_polygon.shp"

out_tif       = "mart_rusle_output/K_US_kffact_0_25cm_UTM14_mart.tif"
pixel_size_m  = 10
target_crs    = "EPSG:26914"    # UTM Zone 14N (meters)
depth_top_cm  = 0.0
depth_bot_cm  = 25.0
# =================================

# ---------- helpers ----------
def _depth_weighted_mu_k(component_df, chorizon_df, k_field, top_cm, bot_cm):
    """
    Build mapunit K (US units) for a given horizon K column (kffact or kwfact):
      horizon overlap-weighted within [top_cm, bot_cm] -> component-weighted by comppct_r -> MUKEY.
    Returns DataFrame: ["mukey","K_US"] where K_US is for the chosen k_field.
    """
    # copy subsets and coerce types
    comp = component_df[["mukey","cokey","comppct_r"]].copy()
    comp["mukey"]     = comp["mukey"].astype(str).str.strip()
    comp["cokey"]     = comp["cokey"].astype(str).str.strip()
    comp["comppct_r"] = pd.to_numeric(comp["comppct_r"], errors="coerce")

    chz = chorizon_df[["cokey","hzdept_r","hzdepb_r", k_field]].copy()
    chz["cokey"]    = chz["cokey"].astype(str).str.strip()
    chz["hzdept_r"] = pd.to_numeric(chz["hzdept_r"], errors="coerce")
    chz["hzdepb_r"] = pd.to_numeric(chz["hzdepb_r"], errors="coerce")
    chz[k_field]    = pd.to_numeric(chz[k_field], errors="coerce")

    # valid horizons and overlap with window
    hz = chz.dropna(subset=["cokey","hzdept_r","hzdepb_r", k_field]).copy()
    hz["overlap_cm"] = np.clip(np.minimum(hz["hzdepb_r"], bot_cm) - np.maximum(hz["hzdept_r"], top_cm), 0, None)
    hz = hz[hz["overlap_cm"] > 0]

    # component-level depth-weighted K
    def _comp_k(g):
        w = g["overlap_cm"].to_numpy()
        k = g[k_field].to_numpy()
        return pd.Series({"K_comp_US": float(np.sum(k*w) / np.sum(w))})

    comp_depth = (
        hz[["cokey","overlap_cm", k_field]]
          .groupby("cokey", group_keys=False)
          .apply(_comp_k)
          .reset_index()
    )

    # component-weighted to MUKEY
    comp_w = (
        comp.merge(comp_depth, on="cokey", how="left")
            .dropna(subset=["mukey","comppct_r","K_comp_US"])
    )
    comp_w = comp_w[comp_w["comppct_r"] > 0]

    def _mu_k(g):
        w = g["comppct_r"].to_numpy()
        k = g["K_comp_US"].to_numpy()
        return pd.Series({"K_US": float(np.sum(k*w) / np.sum(w))})

    mu_k = (
        comp_w[["mukey","comppct_r","K_comp_US"]]
            .groupby("mukey", group_keys=False)
            .apply(_mu_k)
            .reset_index()
    )
    return mu_k  # mukey, K_US

# ---------- load & normalize ----------
component = pd.read_csv(component_csv, dtype=str, low_memory=False)
chorizon  = pd.read_csv(chorizon_csv,  dtype=str, low_memory=False)
component.columns = [c.strip().lower() for c in component.columns]
chorizon.columns  = [c.strip().lower() for c in chorizon.columns]

# Required fields
for c in ["mukey","cokey","comppct_r"]:
    if c not in component.columns:
        raise ValueError(f"component.csv missing: {c}")
if "hzdept_r" not in chorizon.columns:
    raise ValueError("chorizon.csv missing 'hzdept_r'")
if "hzdepb_r" not in chorizon.columns and "hzthk_r" not in chorizon.columns:
    raise ValueError("Need 'hzdepb_r' or 'hzthk_r' in chorizon.csv")
# create hzdepb_r if needed
if "hzdepb_r" not in chorizon.columns:
    chorizon["hzdept_r"] = pd.to_numeric(chorizon["hzdept_r"], errors="coerce")
    chorizon["hzthk_r"]  = pd.to_numeric(chorizon["hzthk_r"], errors="coerce")
    chorizon["hzdepb_r"] = chorizon["hzdept_r"] + chorizon["hzthk_r"]

# Ensure both kffact and kwfact columns exist (we’ll compute whichever is present)
if "kffact" not in chorizon.columns and "kwfact" not in chorizon.columns:
    raise ValueError("chorizon.csv must have at least one of 'kffact' or 'kwfact'")

# ---------- build MU K from kffact (primary) ----------
if "kffact" in chorizon.columns:
    mu_k_kff = _depth_weighted_mu_k(component, chorizon, "kffact", depth_top_cm, depth_bot_cm)
    mu_k_kff = mu_k_kff.rename(columns={"K_US": "K_kffact_US"})
else:
    mu_k_kff = pd.DataFrame(columns=["mukey","K_kffact_US"])

# ---------- build MU K from kwfact (fallback) ----------
if "kwfact" in chorizon.columns:
    mu_k_kw = _depth_weighted_mu_k(component, chorizon, "kwfact", depth_top_cm, depth_bot_cm)
    mu_k_kw = mu_k_kw.rename(columns={"K_US": "K_kwfact_US"})
else:
    mu_k_kw = pd.DataFrame(columns=["mukey","K_kwfact_US"])

# ---------- merge MU tables ----------
mu_merged = pd.merge(mu_k_kff, mu_k_kw, on="mukey", how="outer")

# ---------- join to polygons ----------
gdf = gpd.read_file(aoi_shp)
mu_field = "MUKEY" if "MUKEY" in gdf.columns else ("mukey" if "mukey" in gdf.columns else None)
if mu_field is None:
    raise ValueError("AOI shapefile lacks MUKEY field.")
gdf["MUKEY"] = gdf[mu_field].astype(str).str.strip()

gdf = gdf.merge(mu_merged, left_on="MUKEY", right_on="mukey", how="left").drop(columns=["mukey"])

# ---------- strict WATER identification (for the final zero-fill step only) ----------
if mapunit_csv and os.path.exists(mapunit_csv):
    mu = pd.read_csv(mapunit_csv, dtype=str, low_memory=False)
    mu.columns = [c.strip().lower() for c in mu.columns]
    if "mukey" in mu.columns:
        mu["mukey"] = mu["mukey"].astype(str).str.strip()
        # explicit “water” only (no musym shortcuts)
        name_is_water = mu["muname"].str.contains(r"\bwater\b", case=False, na=False) if "muname" in mu.columns else False
        kind_is_water = mu["mukind"].str.contains("water", case=False, na=False) if "mukind" in mu.columns else False
        mu["_is_water"] = name_is_water | kind_is_water
        gdf = gdf.merge(mu[["mukey","_is_water"]], left_on="MUKEY", right_on="mukey", how="left").drop(columns=["mukey"])
    else:
        gdf["_is_water"] = np.nan
else:
    gdf["_is_water"] = np.nan

# ---------- apply your fill logic ----------
# Start with kffact
gdf["K_US"] = gdf["K_kffact_US"]

#########################
# Track audit categories
audit = pd.DataFrame({"MUKEY": gdf["MUKEY"]})

# Initial category: NaN or kffact
audit["source"] = np.where(
    gdf["K_kffact_US"].notna(), "kffact", "NaN_initial"
)
##################################

# Count NaNs before fill
n_nan_before = int(gdf["K_US"].isna().sum())

# Fill NaNs with kwfact (only where currently NaN)
if "K_kwfact_US" in gdf.columns:
    gdf.loc[gdf["K_US"].isna(), "K_US"] = gdf.loc[gdf["K_US"].isna(), "K_kwfact_US"]

n_filled_kw = int((gdf["K_US"].notna()).sum() - (gdf["K_kffact_US"].notna()).sum())

# Final: where still NaN AND marked water -> set to 0
mask_water_zero = gdf["K_US"].isna() & (gdf["_is_water"] == True)
n_water_zero = int(mask_water_zero.sum())
gdf.loc[mask_water_zero, "K_US"] = 0.0

###################################
audit.loc[mask_water_zero, "source"] = "water_zero"


# Still NaN after all fills
mask_still_nan = gdf["K_US"].isna()
audit.loc[mask_still_nan, "source"] = "still_nan"

# Audit counts
audit_summary = audit["source"].value_counts().rename_axis("source").reset_index(name="count")

print("\n=== AUDIT SUMMARY ===")
for _, row in audit_summary.iterrows():
    print(f"{row['source']}: {row['count']}")
print("=====================\n")
##########################################

print(f"[INFO] Polygons with K_US NaN before fill: {n_nan_before}")
print(f"[INFO] Filled from kwfact: {n_filled_kw}")
print(f"[INFO] Remaining NaN set to 0 because WATER: {n_water_zero}")

# Optional: drop helper columns
for c in ["K_kffact_US","K_kwfact_US","_is_water"]:
    if c in gdf.columns:
        gdf.drop(columns=[c], inplace=True)

# ---------- geometry repair & dissolve ----------
gdf["geometry"] = gdf.geometry.buffer(0)
gdf = gdf.dissolve(by="MUKEY", as_index=False, aggfunc={"K_US":"mean"})

coverage = 100 * (1 - gdf["K_US"].isna().mean())
print(f"[i] Coverage after fills & dissolve: {coverage:.1f}%")

# ---------- project to UTM 14N ----------
if gdf.crs is None:
    gdf = gdf.set_crs("EPSG:4326").to_crs(target_crs)
else:
    gdf = gdf.to_crs(target_crs)
print(f"[i] Working CRS: {gdf.crs}")

# ---------- rasterize manually with rasterio (all_touched=True) ----------
minx, miny, maxx, maxy = gdf.total_bounds
px = float(pixel_size_m)
minx_snapped = math.floor(minx / px) * px
miny_snapped = math.floor(miny / px) * px
maxx_snapped = math.ceil(maxx / px) * px
maxy_snapped = math.ceil(maxy / px) * px

width  = int(round((maxx_snapped - minx_snapped) / px))
height = int(round((maxy_snapped - miny_snapped) / px))
transform = from_origin(minx_snapped, maxy_snapped, px, px)

nodata = -9999.0
dtype  = "float32"

# Only burn geometries with non-NaN K
shapes = [
    (mapping(geom), float(val))
    for geom, val in zip(gdf.geometry, gdf["K_US"])
    if geom is not None and not np.isnan(val)
]

arr = rasterize(
    shapes=shapes,
    out_shape=(height, width),
    transform=transform,
    fill=nodata,
    all_touched=True,
    dtype=dtype,
)

profile = {
    "driver": "GTiff",
    "height": height,
    "width": width,
    "count": 1,
    "dtype": dtype,
    "crs": gdf.crs,
    "transform": transform,
    "compress": "LZW",
    "nodata": nodata,
}
with rasterio.open(out_tif, "w", **profile) as dst:
    dst.write(arr, 1)

print("✅ K raster (US units, kffact with kwfact fallback; water→0 only if still NaN) →", out_tif)

# QA
pct_nodata = 100.0 * np.count_nonzero(arr == nodata) / arr.size
print(f"[QA] NoData pixels: {pct_nodata:.2f}%")

C:\Users\User\AppData\Local\Temp\ipykernel_10800\2942424193.py:71: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_comp_k)
C:\Users\User\AppData\Local\Temp\ipykernel_10800\2942424193.py:90: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_mu_k)
C:\Users\User\AppData\Local\Temp\ipykernel_10800\2942424193.py:71: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behav


=== AUDIT SUMMARY ===
kffact: 69
water_zero: 3

[INFO] Polygons with K_US NaN before fill: 3
[INFO] Filled from kwfact: 0
[INFO] Remaining NaN set to 0 because WATER: 3
[i] Coverage after fills & dissolve: 100.0%
[i] Working CRS: EPSG:26914
✅ K raster (US units, kffact with kwfact fallback; water→0 only if still NaN) → mart_rusle_output/K_US_kffact_0_25cm_UTM14_mart.tif
[QA] NoData pixels: 3.96%
